# 02. 데이터 전처리 및 피처 엔지니어링

EDA에서 발견한 인사이트를 바탕으로, 모델 학습에 적합한 데이터셋을 구축합니다.

## 처리 과정
1. 데이터 로드 및 불필요 컬럼 제거
2. 범주형 변수 인코딩
3. Lag 피처 생성 (시계열 특성 반영)
4. 파생 변수 생성 (비율, 변화율 등)
5. 타겟 변수 생성 (폐업률 → 이진 분류)
6. Train/Val/Test 분할 및 저장

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

: 

## 1. 데이터 로드 및 불필요 컬럼 제거

In [ ]:
df = pd.read_csv('../data/processed/merged_data.csv')
print(f'원본 데이터: {df.shape}')

# 타겟 누수 방지를 위한 컬럼 제거
# - 폐업_점포_수: 타겟(폐업률)과 직접적으로 연관 → 누수
# - 폐업_영업_개월_평균: 이미 폐업한 점포의 정보 → 사전 예측에 부적절
# - 서울시_폐업_영업_개월_평균: 동일 이유
drop_cols = ['폐업_점포_수']

# 실제 존재하는 컬럼만 제거
cols_to_drop = [c for c in drop_cols if c in df.columns]
if '폐업_영업_개월_평균' in df.columns:
    cols_to_drop.append('폐업_영업_개월_평균')
if '서울시_폐업_영업_개월_평균' in df.columns:
    cols_to_drop.append('서울시_폐업_영업_개월_평균')

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f'컬럼 제거 후: {df.shape}')
print(f'제거된 컬럼: {cols_to_drop}')

## 2. 범주형 변수 인코딩

고유값이 많은 범주형 변수(자치구, 업종, 상권변화지표)에 **Label Encoding** 적용합니다.
- One-Hot Encoding 대신 Label Encoding을 사용한 이유: 고유값 수가 많아 차원이 폭발적으로 증가하기 때문

In [ ]:
# 범주형 컬럼 식별
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'범주형 컬럼: {cat_cols}')

# Label Encoding 적용
le_dict = {}  # 나중에 역변환용
encoded_cols = []

for col in cat_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le
    encoded_cols.append(col + '_encoded')
    print(f'  {col}: {len(le.classes_)}개 카테고리 → {col}_encoded')

# 원본 범주형 컬럼 제거
df.drop(columns=cat_cols, inplace=True)
print(f'\n인코딩 후: {df.shape}')

## 3. Lag 피처 생성

시계열 데이터의 특성을 반영하기 위해, 직전 1~2분기의 값을 피처로 활용합니다.
- lag1: 1분기 전 값
- lag2: 2분기 전 값

In [ ]:
# Lag 피처 대상 컬럼 (핵심 수치형 변수들)
lag_target_cols = [
    '당월_매출_금액', '당월_매출_건수', '점포_수', '유사_업종_점포_수',
    '프랜차이즈_점포_수', '개업_률', '개업_점포_수'
]

# 전체임대료, 유동인구, 상주인구, 직장인구도 가능하면 추가
optional_lag_cols = ['전체임대료', '총_유동인구_수', '총_상주인구_수', '총_직장인구_수',
                     '토요일_매출_금액', '일요일_매출_금액', '시간대_21_24_매출_금액',
                     '연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액',
                     '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액']

for col in optional_lag_cols:
    if col in df.columns:
        lag_target_cols.append(col)

# 정렬 (분기 + 자치구 + 업종 기준)
sort_cols = ['기준_년분기_코드']
group_cols = [c for c in encoded_cols if '자치구' in c or '업종' in c]

df.sort_values(by=sort_cols, inplace=True)

# Lag 피처 생성
lag_features = []
for col in lag_target_cols:
    if col in df.columns:
        for lag in [1, 2]:
            lag_col_name = f'{col}_lag{lag}'
            df[lag_col_name] = df.groupby(group_cols)[col].shift(lag)
            lag_features.append(lag_col_name)

print(f'Lag 피처 {len(lag_features)}개 생성')
print(f'데이터 크기: {df.shape}')

## 4. 파생 변수 생성

비즈니스 의미가 있는 파생 피처를 생성합니다.

In [ ]:
# 매출 변화율 (lag1 대비)
if '당월_매출_금액_lag1' in df.columns and '당월_매출_금액_lag2' in df.columns:
    df['매출_변화율'] = (df['당월_매출_금액_lag1'] - df['당월_매출_금액_lag2']) / (df['당월_매출_금액_lag2'] + 1)

# 매출 감소 여부 (이전 분기 대비)
if '당월_매출_금액_lag1' in df.columns and '당월_매출_금액_lag2' in df.columns:
    df['매출_감소'] = (df['당월_매출_금액_lag1'] < df['당월_매출_금액_lag2']).astype(int)

# 점포당 매출
if '당월_매출_금액_lag1' in df.columns and '점포_수_lag1' in df.columns:
    df['점포당_매출'] = df['당월_매출_금액_lag1'] / (df['점포_수_lag1'] + 1)

# 프랜차이즈 비율
if '프랜차이즈_점포_수_lag1' in df.columns and '점포_수_lag1' in df.columns:
    df['프랜차이즈_비율'] = df['프랜차이즈_점포_수_lag1'] / (df['점포_수_lag1'] + 1)

# 경쟁 밀도 (유사 업종 점포 수 / 점포 수)
if '유사_업종_점포_수_lag1' in df.columns and '점포_수_lag1' in df.columns:
    df['경쟁_밀도'] = df['유사_업종_점포_수_lag1'] / (df['점포_수_lag1'] + 1)

# 주말 매출 비율
if '토요일_매출_금액_lag1' in df.columns and '일요일_매출_금액_lag1' in df.columns and '당월_매출_금액_lag1' in df.columns:
    df['주말_매출_비율'] = (df['토요일_매출_금액_lag1'] + df['일요일_매출_금액_lag1']) / (df['당월_매출_금액_lag1'] + 1)

# 야간 매출 비율
if '시간대_21_24_매출_금액_lag1' in df.columns and '당월_매출_금액_lag1' in df.columns:
    df['야간_매출_비율'] = df['시간대_21_24_매출_금액_lag1'] / (df['당월_매출_금액_lag1'] + 1)

print(f'파생 변수 생성 후: {df.shape}')

## 5. 타겟 변수 생성

### 왜 이진 분류로 전환했는가?
- EDA에서 확인한 바와 같이, 폐업률은 심한 우측 꼬리 분포를 가짐
- 회귀 모델로는 일반화 성능이 낮았음 (r² < 0.5) → 03_regression_trial에서 상세 설명
- **"몇 %인지"보다 "위험한지 아닌지"**가 비즈니스적으로 더 유용

### 기준
- 폐업률 **중앙값** 이상 → `closure_risk = 1` (위험)
- 폐업률 중앙값 미만 → `closure_risk = 0` (정상)

In [ ]:
# 타겟 생성
threshold = df['폐업_률'].median()
df['closure_risk'] = (df['폐업_률'] > threshold).astype(int)

print(f'폐업률 중앙값 (threshold): {threshold:.2f}%')
print(f'\n타겟 분포:')
print(df['closure_risk'].value_counts())
print(f'\n비율: {df["closure_risk"].value_counts(normalize=True).to_dict()}')

## 6. 피처 선택 및 Train/Val/Test 분할

In [ ]:
# 피처에서 제외할 컬럼
exclude_cols = ['기준_년분기_코드', '폐업_률', 'closure_risk']

# Lag가 없는 원본 수치형 컬럼도 제외 (미래 정보 누수 방지)
original_numeric = [c for c in lag_target_cols if c in df.columns]
exclude_cols.extend(original_numeric)

# 피처 컬럼 선정
feature_cols = [c for c in df.columns if c not in exclude_cols]

# NaN이 있는 행 제거 (lag로 인해 초기 분기 데이터는 NaN)
df_clean = df.dropna(subset=feature_cols + ['closure_risk']).copy()
print(f'NaN 제거 후: {df_clean.shape}')

X = df_clean[feature_cols]
y = df_clean[['closure_risk']]

print(f'\n최종 피처 수: {X.shape[1]}')
print(f'최종 샘플 수: {X.shape[0]}')
print(f'타겟 분포:\n{y["closure_risk"].value_counts()}')

In [ ]:
# Train / Val / Test 분할 (7:1:2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.125, random_state=42, stratify=y_train
)

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

In [ ]:
# 저장
X_train.to_csv('../data/processed/train_features.csv', index=False)
y_train.to_csv('../data/processed/train_target.csv', index=False)
X_val.to_csv('../data/processed/val_features.csv', index=False)
y_val.to_csv('../data/processed/val_target.csv', index=False)
X_test.to_csv('../data/processed/test_features.csv', index=False)
y_test.to_csv('../data/processed/test_target.csv', index=False)

print('모든 데이터셋 저장 완료!')
print(f'저장 위치: ../data/processed/')
print(f'피처 목록: {feature_cols[:10]}...')

## 정리

### 전처리 파이프라인 요약

| 단계 | 내용 | 비고 |
|------|------|------|
| 1 | 누수 컬럼 제거 | 폐업_점포_수, 폐업_영업_개월_평균 |
| 2 | Label Encoding | 자치구, 업종, 상권변화지표 |
| 3 | Lag 피처 생성 | lag1, lag2 (1~2분기 전 값) |
| 4 | 파생 변수 | 매출변화율, 프랜차이즈비율, 경쟁밀도 등 |
| 5 | 타겟 이진화 | 폐업률 중앙값 기준 |
| 6 | 데이터 분할 | Train 70% / Val 10% / Test 20% |

→ 다음: `03_regression_trial.ipynb`에서 회귀 모델 시도 및 분류 전환 근거 설명